In [7]:
import json
import os
import shutil
from pathlib import Path

import yaml
from pyoxigraph import Store, RdfFormat, parse
from pyshacl import validate


SRC_DIR = Path(".")
DATA_DIR = Path("data")
BUILD_DIR = Path("build")


def yaml2json(yaml_file: Path) -> Path:
    json_file = BUILD_DIR / f"{yaml_file.stem}.jsonld"

    os.makedirs(BUILD_DIR, exist_ok=True)

    with yaml_file.open("rb") as f:
        data = yaml.safe_load(f)

    with open(json_file, "w") as g:
        json.dump(data, g, default=str, indent=2)

    return json_file


shutil.rmtree(BUILD_DIR, ignore_errors=True)
nbnl_eq_profile_jsonld = yaml2json(SRC_DIR / "nbnl-equipment.prof.shacl.yamlld")

data_graph = Store()
with open(DATA_DIR / "equipment_four_stations_EQ.jsonld", "rb") as f:
    data_graph.bulk_load(f.read(), format=RdfFormat.JSON_LD)

shapes_graph = Store()
with open(nbnl_eq_profile_jsonld, "rb") as f:
    shapes_graph.bulk_load(f.read(), format=RdfFormat.JSON_LD)

conforms, results_graph, results_text = validate(
    data_graph,
    shacl_graph_format=RdfFormat.JSON_LD,
    shacl_graph=shapes_graph,
    advanced=True,
)


# data_dataset = Dataset()
# with open(
#     DATADIR / "AS_EQ_OP_Westland_FGU" / "equipment_four_stations_EQ.jsonld", "rb"
# ) as f:
#     data_dataset.bulk_load(f.read, format=RdfFormat.JSON_LD)

# shapes_dataset = Dataset()
# with open(BUILDDIR / "nbnl-equipment.shacl.jsonld", "rb") as f:
#     shapes_dataset.bulk_load(f.read(), format=RdfFormat.JSON_LD)

# ontology_graph = Dataset()
# with open(BUILDDIR / "nbnl-equipment.rdfs.jsonld", "rb") as f:
#     ontology_graph.bulk_load(f.read(), format=RdfFormat.JSON_LD)

# conforms, results_graph, results_text = validate(
#     data_graph=data_dataset,
#     shacl_graph=shapes_graph,
#     ont_graph=ontology_graph,  # TODO: This is very specific to this validator. Can we do this differently?
#     meta_shacl=True,
#     inference="rdfs",
#     do_owl_imports=True,
# )

# print(results_text)

ValueError: Cannot determine the format of the input graph